# Дендропарк імені Василя Байтали очима супутника і штучного інтелекту

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yevheniikhan/lubny-dendropark-ai/blob/main/notebooks/dendropark_ai_gis.ipynb)

Лубенський лісотехнічний фаховий коледж · практичне заняття з ГІС і ДЗЗ

За 20 хвилин ви:

1. побачите свій дендропарк на карті з роздільною здатністю менше метра;
2. дізнаєтеся висоту кожної ділянки намету, виміряну з космосу;
3. подивитеся, як парк «дихає» протягом року за знімками Sentinel-2;
4. дасте штучному інтелекту знайти зони парку без жодної підказки;
5. порахуєте окремі крони дерев (бонус із GPU).

Потрібен лише обліковий запис Google. Перед стартом: **Середовище виконання → Змінити тип → CPU** (для кроку 5 – GPU).

## 0. Підготовка

Вкажіть свій проєкт Google Cloud з увімкненим Earth Engine (безкоштовно для навчання: [earthengine.google.com/noncommercial](https://earthengine.google.com/noncommercial/)). Вікно авторизації підтверджуєте ви.

In [ ]:
!pip -q install geemap pyproj scikit-learn
import ee, geemap, numpy as np, requests, matplotlib.pyplot as plt
PROJECT = 'your-ee-project'          # <- замініть на свій проєкт
try:
    ee.Initialize(project=PROJECT)
except Exception:
    ee.Authenticate(); ee.Initialize(project=PROJECT)

# Дендропарк імені В. Д. Байтали, Лубни. Межа – картосхема з буклета коледжу, прив'язана до знімка
# афінним перетворенням за 12 опорними точками (data/lltk/gcp.json у репозиторії).
PARK = ee.Geometry.Polygon([[32.985061, 49.984098], [32.984749, 49.984147], [32.984898, 49.984423], [32.985223, 49.984623], [32.985362, 49.984744], [32.985651, 49.984922], [32.986004, 49.985082], [32.986127, 49.985176], [32.986286, 49.985246], [32.986468, 49.985403], [32.986805, 49.985489], [32.987029, 49.985904], [32.987015, 49.985923], [32.987055, 49.986133], [32.987108, 49.98623], [32.987126, 49.986399], [32.98719, 49.986483], [32.987173, 49.986576], [32.987235, 49.98678], [32.987385, 49.986789], [32.987735, 49.986855], [32.987828, 49.986847], [32.98805, 49.9869], [32.98845, 49.986923], [32.988769, 49.987155], [32.989002, 49.987363], [32.989126, 49.987415], [32.989169, 49.987493], [32.989517, 49.987734], [32.989626, 49.987847], [32.989807, 49.987958], [32.990004, 49.988143], [32.990061, 49.98816], [32.99055, 49.988034], [32.990734, 49.98797], [32.992157, 49.987602], [32.992171, 49.987583], [32.993021, 49.987363], [32.993049, 49.987325], [32.993162, 49.987311], [32.993886, 49.987124], [32.994387, 49.986978], [32.994397, 49.986771], [32.994356, 49.986561], [32.994372, 49.986321], [32.994309, 49.98607], [32.994333, 49.98589], [32.994253, 49.985832], [32.994323, 49.985782], [32.99431, 49.985534], [32.994273, 49.985507], [32.994271, 49.985103], [32.994178, 49.985064], [32.994268, 49.984962], [32.99422, 49.984738], [32.99405, 49.984782], [32.99382, 49.984715], [32.993841, 49.98471], [32.993867, 49.984577], [32.994058, 49.984528], [32.994229, 49.984531], [32.994171, 49.984261], [32.994207, 49.984221], [32.994188, 49.984006], [32.994157, 49.983948], [32.994157, 49.983636], [32.994119, 49.983564], [32.994123, 49.983078], [32.994081, 49.983], [32.994101, 49.982768], [32.994063, 49.982652], [32.99407, 49.982351], [32.994031, 49.982235], [32.994022, 49.98186], [32.993964, 49.981796], [32.992901, 49.982071], [32.992675, 49.982145], [32.99208, 49.982299], [32.992005, 49.982148], [32.991956, 49.982122], [32.99186, 49.981973], [32.991841, 49.981977], [32.99179, 49.98184], [32.991739, 49.981812], [32.991634, 49.981644], [32.991614, 49.981647], [32.991327, 49.981176], [32.991275, 49.981155], [32.990921, 49.981232], [32.990958, 49.981321], [32.990771, 49.981383], [32.990497, 49.981428], [32.99037, 49.981485], [32.989645, 49.981616], [32.989604, 49.981651], [32.989717, 49.981767], [32.989691, 49.981771], [32.989697, 49.981784], [32.98914, 49.981861], [32.988911, 49.981907], [32.988904, 49.982015], [32.988986, 49.98211], [32.989055, 49.982302], [32.989183, 49.982481], [32.989154, 49.982486], [32.989428, 49.983], [32.988561, 49.983224], [32.988547, 49.983244], [32.988341, 49.983265], [32.985642, 49.983963], [32.985635, 49.983949], [32.985061, 49.984098]])
CENTER = PARK.centroid(1).coordinates().getInfo()[::-1]
UTM = 'EPSG:32636'                    # метрична проєкція для зони 36N
print('Площа контуру, га:', round(PARK.area(1).getInfo() / 1e4, 1))

## 1. Дендропарк з висоти 500 м

Підкладка – мозаїка аерофото- і супутникових знімків Esri з роздільною здатністю до 0,5 м. Увімкніть шари в правому верхньому куті карти. Інструментом малювання можна обвести власний контур парку: він стане доступним як `m.user_roi`.

In [ ]:
m = geemap.Map(center=CENTER, zoom=16, basemap='Esri.WorldImagery')
m.addLayer(ee.Image().paint(PARK, 0, 3), {'palette': ['ffd400']}, 'Контур дендропарку')
m

## 2. Висота дерев, виміряна з космосу

Карта висоти намету Meta і World Resources Institute (роздільна здатність 1 м) отримана нейромережею, навченою на лідарних вимірах. Подивимося, де ростуть найвищі дерева парку.

In [ ]:
col = ee.ImageCollection('projects/meta-forest-monitoring-okw37/assets/CanopyHeight').filterBounds(PARK)
chm = col.mosaic().select(0).rename('h').setDefaultProjection(col.first().projection())
vis_h = {'min': 0, 'max': 30, 'palette': ['ffffff', 'e8d9a8', 'b9cf96', '7fae86', '3f7d66', '14524a', '0a2f2a']}
m = geemap.Map(center=CENTER, zoom=16, basemap='Esri.WorldImagery')
m.addLayer(chm.updateMask(chm.gt(2)).clip(PARK), vis_h, 'Висота намету, м')
m.add_colorbar(vis_h, label='Висота намету, м')
m

In [ ]:
# Статистика висот у межах парку
st = chm.reduceRegion(ee.Reducer.percentile([50, 90, 99]).combine(ee.Reducer.max(), sharedInputs=True), PARK, 1, maxPixels=1e9).getInfo()
print({k: round(v, 1) for k, v in st.items()})
box = PARK.buffer(60).bounds(1, ee.Projection(UTM))
h2 = chm.unmask(0).reduceResolution(ee.Reducer.mean(), maxPixels=16).reproject(crs=UTM, scale=2)
arr = np.array(h2.sampleRectangle(region=box, defaultValue=0).get('h').getInfo(), float)
plt.figure(figsize=(6, 3)); plt.hist(arr[arr > 2].ravel(), bins=40, color='#14524A')
plt.xlabel('висота, м'); plt.ylabel('кількість пікселів 2 × 2 м'); plt.title('Розподіл висот намету в дендропарку'); plt.show()

### 2a. Дендропарк у 3D

Перетворимо карту висот на тривимірну модель і обертатимемо її.

In [ ]:
from matplotlib import animation
from IPython.display import HTML
z = arr[::2, ::2].astype(float); z[z < 0] = 0
y, x = np.mgrid[0:z.shape[0], 0:z.shape[1]]
fig = plt.figure(figsize=(7, 5)); ax = fig.add_subplot(projection='3d')
ax.plot_surface(x, -y, z, cmap='YlGn', linewidth=0, antialiased=False, vmin=0, vmax=30)
ax.set_zlim(0, 60); ax.axis('off')
anim = animation.FuncAnimation(fig, lambda a: ax.view_init(35, a), frames=range(0, 360, 4), interval=60)
plt.close(); HTML(anim.to_jshtml())

## 3. Парк дихає: рік очима Sentinel-2

NDVI – індекс, що показує активність зеленого листя. Порахуємо його для кожного місяця 2024 року і зробимо анімацію.

In [ ]:
csp = ee.ImageCollection('GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED')     # маска хмар Cloud Score+
s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED').filterBounds(PARK).filterDate('2024-01-01', '2025-01-01')
      .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 80)).linkCollection(csp, ['cs_cdf'])
      .map(lambda i: i.updateMask(i.select('cs_cdf').gte(0.6))))
def monthly(mn):
    c = s2.filter(ee.Filter.calendarRange(mn, mn, 'month'))
    nd = c.median().normalizedDifference(['B8', 'B4']).rename('NDVI')
    return ee.Image(ee.Algorithms.If(c.size().gt(0), nd, ee.Image.constant(0).rename('NDVI').updateMask(0))).set('month', mn)
months = ee.ImageCollection([monthly(mn) for mn in range(1, 13)])
curve = [months.filter(ee.Filter.eq('month', mn)).first().reduceRegion(ee.Reducer.mean(), PARK, 10).get('NDVI').getInfo() for mn in range(1, 13)]
plt.figure(figsize=(6, 3)); plt.plot(range(1, 13), curve, 'o-', color='#14524A')
plt.xticks(range(1, 13), ['січ', 'лют', 'бер', 'кві', 'тра', 'чер', 'лип', 'сер', 'вер', 'жов', 'лис', 'гру'])
plt.ylabel('середній NDVI'); plt.title('Річний цикл дендропарку, 2024'); plt.grid(alpha=.3); plt.show()

In [ ]:
region = PARK.buffer(150).bounds(1, ee.Projection(UTM))
vis_n = {'min': 0, 'max': 0.9, 'palette': ['8c510a', 'd8b365', 'f6e8c3', 'c7eae5', '5ab4ac', '01665e']}
frames = months.map(lambda i: i.visualize(**vis_n))
url = frames.getVideoThumbURL({'region': region, 'dimensions': 480, 'crs': UTM, 'framesPerSecond': 2})
from IPython.display import Image
Image(url=url)

## 4. Штучний інтелект знаходить зони парку сам

Модель AlphaEarth Foundations (Google DeepMind) стиснула річні спостереження різних супутників у 64 числа для кожного пікселя 10 м. Ми не даємо моделі жодних назв порід: алгоритм k-середніх лише групує подібні пікселі. Порівняйте зони з тим, що знаєте про парк.

In [ ]:
emb = (ee.ImageCollection('GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL')
       .filterDate('2024-01-01', '2025-01-01').filterBounds(PARK).mosaic())
sample = emb.sample(region=PARK, scale=10, numPixels=3000, seed=1)
K = 5                                  # спробуйте 3, 5, 8
clusterer = ee.Clusterer.wekaKMeans(K, seed=1).train(sample)
zones = emb.cluster(clusterer).clip(PARK)
m = geemap.Map(center=CENTER, zoom=16, basemap='Esri.WorldImagery')
m.addLayer(zones.randomVisualizer(), {}, f'{K} зон, знайдених без учителя', opacity=0.7)
m

## 5. Бонус: штучний інтелект рахує крони (потрібен GPU)

Модель сегментації SAM за текстовою підказкою «tree» виділяє окремі крони на знімку високої роздільності. Результат перевіряйте очима: модель навчена на звичайних фото, тому може помилятися в густому наметі.

In [ ]:
# Змініть середовище на GPU і виконайте:
# !pip -q install segment-geospatial groundingdino-py
# from samgeo import tms_to_geotiff
# from samgeo.text_sam import LangSAM
# b = PARK.bounds().coordinates().getInfo()[0]
# bbox = [b[0][0], b[0][1], b[2][0], b[2][1]]
# tms_to_geotiff(output='park.tif', bbox=bbox, zoom=19, source='Satellite', overwrite=True)
# sam = LangSAM()
# sam.predict('park.tif', 'tree', box_threshold=0.24, text_threshold=0.24)
# sam.show_anns(cmap='Greens', add_boxes=False, alpha=0.6, title='Крони, знайдені SAM')
# sam.raster_to_vector('park_mask.tif', 'tree_crowns.gpkg')   # полігони для QGIS

## Далі

У цьому ноутбуці ви пройшли шлях фахівця з дистанційного моніторингу лісів: межа парку, висота крон, сезонна динаміка і зони, знайдені штучним інтелектом. Цих методів навчають у Навчально-науковому інституті лісового і садово-паркового господарства НУБіП України (nubip.edu.ua).

**Джерела даних:** Esri World Imagery; Meta & WRI Global Canopy Height (Tolan et al., 2024); Copernicus Sentinel-2; Cloud Score+ (Pasquarella et al., 2023); AlphaEarth Foundations Satellite Embedding (Brown et al., 2025); SAM (Meta AI). Межа парку – картосхема з буклета коледжу (lltk.edu.ua/pro-koledzh/dendropark).